In [2]:
from pathlib import Path
import sys

import certifi
import requests

from azure.core.credentials import AzureNamedKeyCredential
from azure.storage.blob import BlobServiceClient
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


OVERWRITE = True

ARQUIVOS = [
    {
        "nome": "pib_municipal.json",
        "url": (
            "https://apisidra.ibge.gov.br/"
            "values/t/5938/n6/all/"
            "v/37,516,520,528,6574/p/2021"
        ),
    },
    {
        "nome": "censo_2022.json",
        "url": (
            "https://apisidra.ibge.gov.br/"
            "values/t/4714/n6/all/"
            "v/93,6318,614/p/2022"
        ),
    },
]


def detectar_databricks():
    try:
        dbutils.secrets
        return True
    except (NameError, AttributeError):
        return False


def localizar_raiz_projeto():
    atual = Path.cwd().resolve()

    for candidato in [atual, *atual.parents]:
        if (candidato / "src").is_dir():
            return candidato

    raise FileNotFoundError(
        "Pasta 'src' não encontrada no diretório atual "
        "ou nos diretórios pais."
    )


def exigir_valor(valor, nome):
    valor = (
        str(valor).strip()
        if valor is not None
        else ""
    )

    if not valor:
        raise RuntimeError(
            f"{nome} não configurado."
        )

    return valor


IS_DATABRICKS = detectar_databricks()
BASE_DIR = localizar_raiz_projeto()

if str(BASE_DIR) not in sys.path:
    sys.path.insert(
        0,
        str(BASE_DIR),
    )

if not IS_DATABRICKS:
    from dotenv import load_dotenv

    ENV_PATH = BASE_DIR / ".env"

    if not ENV_PATH.is_file():
        raise FileNotFoundError(
            f"Arquivo .env não encontrado: "
            f"{ENV_PATH}"
        )

    ambiente_carregado = load_dotenv(
        dotenv_path=ENV_PATH,
        override=True,
    )

    if not ambiente_carregado:
        raise RuntimeError(
            f"Não foi possível carregar: "
            f"{ENV_PATH}"
        )

from src.config.secrets import get_secret


AZURE_STORAGE_ACCOUNT = exigir_valor(
    get_secret(
        "AZURE-STORAGE-ACCOUNT",
        "AZURE_STORAGE_ACCOUNT",
    ),
    "AZURE_STORAGE_ACCOUNT",
)

AZURE_STORAGE_KEY = exigir_valor(
    get_secret(
        "AZURE-STORAGE-KEY",
        "AZURE_STORAGE_KEY",
    ),
    "AZURE_STORAGE_KEY",
)

BRONZE_CONTAINER = exigir_valor(
    get_secret(
        "AZURE-CONTAINER-BRONZE",
        "AZURE_CONTAINER_BRONZE",
    ),
    "AZURE_CONTAINER_BRONZE",
)

LOCAL_BRONZE_PATH = (
    BASE_DIR
    / "dados"
    / "bronze"
)

LOCAL_BRONZE_PATH.mkdir(
    parents=True,
    exist_ok=True,
)


def validar_configuracao():
    if not ARQUIVOS:
        raise ValueError(
            "Nenhum arquivo foi configurado."
        )

    nomes = set()

    for indice, item in enumerate(
        ARQUIVOS,
        start=1,
    ):
        nome = str(
            item.get("nome", "")
        ).strip()

        url = str(
            item.get("url", "")
        ).strip()

        if not nome:
            raise ValueError(
                f"Entrada {indice} sem nome."
            )

        if not url:
            raise ValueError(
                f"Entrada {indice} sem URL."
            )

        if nome in nomes:
            raise ValueError(
                f"Nome duplicado: {nome}"
            )

        if not url.startswith(
            ("http://", "https://")
        ):
            raise ValueError(
                f"URL inválida para {nome}: "
                f"{url}"
            )

        nomes.add(nome)


def criar_sessao_http():
    retry = Retry(
        total=5,
        connect=5,
        read=5,
        status=5,
        backoff_factor=2,
        status_forcelist=[
            429,
            500,
            502,
            503,
            504,
        ],
        allowed_methods=frozenset([
            "GET",
        ]),
        raise_on_status=False,
    )

    adapter = HTTPAdapter(
        max_retries=retry
    )

    session = requests.Session()

    session.mount(
        "https://",
        adapter,
    )

    session.mount(
        "http://",
        adapter,
    )

    session.headers.update({
        "User-Agent": (
            "Mozilla/5.0 "
            "IBGE-Azure-Pipeline/1.0"
        ),
        "Accept": "application/json,*/*",
        "Connection": "close",
    })

    return session


def criar_container_client():
    credential = AzureNamedKeyCredential(
        name=AZURE_STORAGE_ACCOUNT,
        key=AZURE_STORAGE_KEY,
    )

    account_url = (
        f"https://{AZURE_STORAGE_ACCOUNT}"
        ".blob.core.windows.net"
    )

    blob_service_client = BlobServiceClient(
        account_url=account_url,
        credential=credential,
    )

    blob_service_client.get_account_information()

    container_client = (
        blob_service_client.get_container_client(
            container=BRONZE_CONTAINER
        )
    )

    container_client.get_container_properties()

    print(
        "✅ Cliente Azure Blob autenticado."
    )

    print(
        "✅ Ambiente:",
        "Databricks"
        if IS_DATABRICKS
        else "local",
    )

    print(
        f"✅ Storage Account: "
        f"{AZURE_STORAGE_ACCOUNT}"
    )

    print(
        f"✅ Contêiner Bronze: "
        f"{BRONZE_CONTAINER}"
    )

    print(
        f"✅ Pasta Bronze local: "
        f"{LOCAL_BRONZE_PATH.resolve()}"
    )

    return container_client


def baixar_arquivo(
    session,
    url,
    destino,
):
    destino = Path(destino)

    destino.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    arquivo_parcial = destino.with_suffix(
        destino.suffix + ".part"
    )

    if arquivo_parcial.exists():
        arquivo_parcial.unlink()

    try:
        with session.get(
            url=url,
            stream=True,
            verify=certifi.where(),
            timeout=(60, 3600),
            allow_redirects=True,
        ) as response:
            response.raise_for_status()

            content_type = (
                response.headers
                .get("Content-Type", "")
                .lower()
            )

            if "text/html" in content_type:
                raise RuntimeError(
                    "O servidor retornou HTML "
                    "em vez do arquivo esperado. "
                    f"URL final: {response.url}"
                )

            with arquivo_parcial.open(
                mode="wb"
            ) as arquivo:
                for chunk in response.iter_content(
                    chunk_size=1024 * 1024
                ):
                    if chunk:
                        arquivo.write(chunk)

        if not arquivo_parcial.is_file():
            raise RuntimeError(
                "Arquivo parcial não foi criado."
            )

        tamanho = (
            arquivo_parcial.stat().st_size
        )

        if tamanho <= 0:
            raise RuntimeError(
                "O download produziu "
                "um arquivo vazio."
            )

        arquivo_parcial.replace(
            destino
        )

        print(
            f"✅ Download concluído: "
            f"{destino.name}"
        )

        print(
            f"✅ Arquivo local: "
            f"{destino.resolve()}"
        )

        print(
            f"✅ Tamanho local: "
            f"{tamanho / 1024:.2f} KB"
        )

        return destino

    except Exception:
        if arquivo_parcial.exists():
            arquivo_parcial.unlink()

        raise


def enviar_para_bronze(
    container_client,
    arquivo_local,
):
    arquivo_local = Path(
        arquivo_local
    )

    if not arquivo_local.is_file():
        raise FileNotFoundError(
            f"Arquivo local não encontrado: "
            f"{arquivo_local}"
        )

    tamanho_local = (
        arquivo_local.stat().st_size
    )

    if tamanho_local <= 0:
        raise RuntimeError(
            f"Arquivo local vazio: "
            f"{arquivo_local}"
        )

    blob_name = arquivo_local.name

    blob_client = (
        container_client.get_blob_client(
            blob=blob_name
        )
    )

    with arquivo_local.open(
        mode="rb"
    ) as arquivo:
        blob_client.upload_blob(
            data=arquivo,
            overwrite=OVERWRITE,
        )

    propriedades = (
        blob_client.get_blob_properties()
    )

    if propriedades.size != tamanho_local:
        raise RuntimeError(
            "O tamanho enviado nãocorresponde "
            "ao arquivo local. "
            f"Local: {tamanho_local} bytes. "
            f"Azure: {propriedades.size} bytes."
        )

    print(
        f"✅ Upload concluído: "
        f"{blob_name}"
    )

    print(
        f"✅ Tamanho no Azure: "
        f"{propriedades.size / 1024:.2f} KB"
    )

    return {
        "blob": blob_name,
        "arquivo_local": str(
            arquivo_local.resolve()
        ),
        "size_bytes": propriedades.size,
        "etag": str(
            propriedades.etag
        ),
        "last_modified": str(
            propriedades.last_modified
        ),
    }


def executar_pipeline():
    validar_configuracao()

    container_client = (
        criar_container_client()
    )

    session = criar_sessao_http()

    uploads_concluidos = []
    falhas = []

    try:
        total = len(ARQUIVOS)

        for indice, item in enumerate(
            ARQUIVOS,
            start=1,
        ):
            nome = item["nome"]
            url = item["url"]

            destino_local = (
                LOCAL_BRONZE_PATH
                / nome
            )

            print()
            print(
                f"[{indice}/{total}] "
                f"Processando: {nome}"
            )

            try:
                arquivo_baixado = baixar_arquivo(
                    session=session,
                    url=url,
                    destino=destino_local,
                )

                resultado = enviar_para_bronze(
                    container_client=container_client,
                    arquivo_local=arquivo_baixado,
                )

                uploads_concluidos.append({
                    "arquivo": nome,
                    "origem": url,
                    "container": BRONZE_CONTAINER,
                    **resultado,
                })

            except Exception as exc:
                erro = (
                    f"{type(exc).__name__}: "
                    f"{exc}"
                )

                falhas.append({
                    "arquivo": nome,
                    "origem": url,
                    "blob": nome,
                    "erro": erro,
                })

                print(
                    f"❌ Falha em {nome}: "
                    f"{erro}"
                )

    finally:
        session.close()

    print()
    print(
        "========== RESUMO =========="
    )

    print(
        f"✅ Arquivos enviados: "
        f"{len(uploads_concluidos)}"
    )

    print(
        f"❌ Falhas: "
        f"{len(falhas)}"
    )

    print(
        f"✅ Contêiner: "
        f"{BRONZE_CONTAINER}"
    )

    print(
        f"✅ Pasta local: "
        f"{LOCAL_BRONZE_PATH.resolve()}"
    )

    if falhas:
        detalhes = "\n".join(
            (
                f"- Arquivo: {falha['arquivo']}\n"
                f"  Origem: {falha['origem']}\n"
                f"  Blob: {falha['blob']}\n"
                f"  Erro: {falha['erro']}"
            )
            for falha in falhas
        )

        raise RuntimeError(
            f"{len(falhas)} arquivo(s) "
            "não foram enviados:\n"
            f"{detalhes}"
        )

    return (
        container_client,
        uploads_concluidos,
    )


def validar_bronze(
    container_client,
    uploads_concluidos,
):
    print()
    print(
        "========== VALIDAÇÃO BRONZE =========="
    )

    for upload in uploads_concluidos:
        arquivo_local = Path(
            upload["arquivo_local"]
        )

        blob_name = upload["blob"]

        if not arquivo_local.is_file():
            raise FileNotFoundError(
                f"Arquivo local não encontrado: "
                f"{arquivo_local}"
            )

        tamanho_local = (
            arquivo_local.stat().st_size
        )

        propriedades = (
            container_client
            .get_blob_client(
                blob=blob_name
            )
            .get_blob_properties()
        )

        if propriedades.size != tamanho_local:
            raise RuntimeError(
                f"Tamanho divergente em "
                f"{blob_name}. "
                f"Local: {tamanho_local}. "
                f"Azure: {propriedades.size}."
            )

        print(
            f"✅ Local: "
            f"{arquivo_local.name} "
            f"({tamanho_local / 1024:.2f} KB)"
        )

        print(
            f"✅ Azure: "
            f"{blob_name} "
            f"({propriedades.size / 1024:.2f} KB)"
        )

    blobs_enviados = {
        upload["blob"]
        for upload in uploads_concluidos
    }

    blobs_na_raiz = [
        blob
        for blob in container_client.list_blobs()
        if "/" not in blob.name
    ]

    print()

    print(
        f"✅ Contêiner validado: "
        f"{BRONZE_CONTAINER}"
    )

    print(
        f"✅ Arquivos desta execução: "
        f"{len(uploads_concluidos)}"
    )

    print(
        f"✅ Blobs na raiz do Bronze: "
        f"{len(blobs_na_raiz)}"
    )

    nao_encontrados = (
        blobs_enviados
        - {
            blob.name
            for blob in blobs_na_raiz
        }
    )

    if nao_encontrados:
        raise RuntimeError(
            "Blobs não encontrados na raiz "
            f"do contêiner: "
            f"{sorted(nao_encontrados)}"
        )

    print(
        "✅ Arquivos locais e blobs "
        "validados com sucesso."
    )


(
    container_client,
    uploads_concluidos,
) = executar_pipeline()

validar_bronze(
    container_client=container_client,
    uploads_concluidos=uploads_concluidos,
)

✅ Cliente Azure Blob autenticado.
✅ Ambiente: local
✅ Storage Account: stfiaptechchallenge2iast
✅ Contêiner Bronze: bronze
✅ Pasta Bronze local: /home/linux/projetos/postech-2iast-fase3-modelo-preditivo-g28/dados/bronze

[1/2] Processando: pib_municipal.json
✅ Download concluído: pib_municipal.json
✅ Arquivo local: /home/linux/projetos/postech-2iast-fase3-modelo-preditivo-g28/dados/bronze/pib_municipal.json
✅ Tamanho local: 10191.82 KB
✅ Upload concluído: pib_municipal.json
✅ Tamanho no Azure: 10191.82 KB

[2/2] Processando: censo_2022.json
✅ Download concluído: censo_2022.json
✅ Arquivo local: /home/linux/projetos/postech-2iast-fase3-modelo-preditivo-g28/dados/bronze/censo_2022.json
✅ Tamanho local: 4542.91 KB
✅ Upload concluído: censo_2022.json
✅ Tamanho no Azure: 4542.91 KB

========== RESUMO ==========
✅ Arquivos enviados: 2
❌ Falhas: 0
✅ Contêiner: bronze
✅ Pasta local: /home/linux/projetos/postech-2iast-fase3-modelo-preditivo-g28/dados/bronze

========== VALIDAÇÃO BRONZE ========